## Tools
### Models can request to call tools that perform tasks such as fetching data from database, searching on web, or running code. Tools are pairing of:
### 1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
### 2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['COHERE_API_KEY'] = os.getenv('COHERE_API_KEY')

llm = init_chat_model('gemini-2.5-flash-lite', model_provider='google_genai')

llm.invoke(["Hello, how are you?"])

AIMessage(content="Hello! I'm doing well, thank you for asking. As a large language model, I don't experience emotions or physical sensations, but I'm functioning optimally and ready to assist you.\n\nHow are *you* doing today? Is there anything I can help you with?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c087b-e1ba-77f0-ad4d-a6bd4b4d045e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 59, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}})

In [3]:
from langchain.tools import tool

@tool
def get_weather(city:str) -> str:
    """Get the weather for a city"""
    return f"The weather in {city} is sunny"

In [5]:
model_with_tools = llm.bind_tools([get_weather])
response = model_with_tools.invoke("What is the weather like in Dadu?")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Dadu"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0883-3ef0-7850-8f44-c3930599a49c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Dadu'}, 'id': '240daed8-cc29-499b-9fd3-1f954b0c8d25', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 16, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}})

### Tool execution loops

In [10]:
# Step 1: Model generates tool calls
messages = [{'role':'user', 'content': 'What is weather like in New York today?'}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tool calls and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response)

content='The weather in New York is sunny.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c088f-b64d-71e2-a9cd-ad4db6f2a22d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 85, 'output_tokens': 8, 'total_tokens': 93, 'input_token_details': {'cache_read': 0}}
